# CSCI 7400 Assignment 7 by David Jean

## Read in data and check the head

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col

# Start Spark session
spark = SparkSession.builder \
    .appName("SentimentClassification") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

#Load the raw text file (
raw_df = spark.read.text("amazon_cells_labelled.txt")

# Split into sentence and label
df = raw_df.select(
    split(col("value"), "\t").getItem(0).alias("sentence"),
    split(col("value"), "\t").getItem(1).cast("int").alias("label")
)

# Show a few rows to confirm
df.show(5, truncate=False)


your 131072x1 screen size is bogus. expect trouble
25/04/03 18:06:22 WARN Utils: Your hostname, DESKTOP-BOJPASN resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/04/03 18:06:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/03 18:06:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/03 18:06:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+----------------------------------------------------------------------------------+-----+
|sentence                                                                          |label|
+----------------------------------------------------------------------------------+-----+
|So there is no way for me to plug it in here in the US unless I go by a converter.|0    |
|Good case, Excellent value.                                                       |1    |
|Great for the jawbone.                                                            |1    |
|Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!   |0    |
|The mic is great.                                                                 |1    |
+----------------------------------------------------------------------------------+-----+
only showing top 5 rows



## Preprocess the data by removing special characters, numbers, and other noise

In [2]:
from pyspark.sql.functions import lower, regexp_replace

#Clean the text: lowercase + remove special chars/numbers
cleaned_df = df.select(
    lower(col("sentence")).alias("sentence"),
    col("label")
).withColumn(
    "sentence",
    regexp_replace("sentence", r"[^a-z\s]", "")  # keep only letters and spaces
)

cleaned_df.show(5, truncate=False)


+---------------------------------------------------------------------------------+-----+
|sentence                                                                         |label|
+---------------------------------------------------------------------------------+-----+
|so there is no way for me to plug it in here in the us unless i go by a converter|0    |
|good case excellent value                                                        |1    |
|great for the jawbone                                                            |1    |
|tied to charger for conversations lasting more than  minutesmajor problems       |0    |
|the mic is great                                                                 |1    |
+---------------------------------------------------------------------------------+-----+
only showing top 5 rows



## Tokenize the data; Remove stop words; Apply word2vec to convert tokens to numerical feature vectors. 

In [3]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
tokenized_df = tokenizer.transform(cleaned_df)

tokenized_df.select("sentence", "words", "label").show(5, truncate=False)


+---------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------+-----+
|sentence                                                                         |words                                                                                                  |label|
+---------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------+-----+
|so there is no way for me to plug it in here in the us unless i go by a converter|[so, there, is, no, way, for, me, to, plug, it, in, here, in, the, us, unless, i, go, by, a, converter]|0    |
|good case excellent value                                                        |[good, case, excellent, value]                                                                         |1    |
|great for the jawbone        

In [4]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(inputCol="words", outputCol="filtered")
filtered_df = remover.transform(tokenized_df)

filtered_df.select("words", "filtered", "label").show(5, truncate=False)


+-------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------+-----+
|words                                                                                                  |filtered                                                         |label|
+-------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------+-----+
|[so, there, is, no, way, for, me, to, plug, it, in, here, in, the, us, unless, i, go, by, a, converter]|[way, plug, us, unless, go, converter]                           |0    |
|[good, case, excellent, value]                                                                         |[good, case, excellent, value]                                   |1    |
|[great, for, the, jawbone]                                                                             |[grea

In [5]:
from pyspark.ml.feature import Word2Vec

word2vec = Word2Vec(vectorSize=100, minCount=1, inputCol="filtered", outputCol="features")
model = word2vec.fit(filtered_df)
vectorized_df = model.transform(filtered_df)

# Final dataset ready for training
vectorized_df.select("filtered", "features", "label").show(5, truncate=False)


25/04/03 18:06:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+-----------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Implement a classification algorithm of your choice using spark MLlib. (The data should be split into train and test), Train the model on the training dataset

In [6]:
from pyspark.ml.classification import LogisticRegression

# 🔹 Split into train/test
train_df, test_df = vectorized_df.randomSplit([0.8, 0.2], seed=42)

# 🔹 Initialize logistic regression
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)

# 🔹 Train the model
lr_model = lr.fit(train_df)





## Use the trained model to make predictions on the test dataset.

In [7]:
#  Full prediction output: sentence, true label, prediction, and probability
# Show full prediction output
predictions = lr_model.transform(test_df)
predictions.select("filtered", "label", "prediction", "probability").show(10, truncate=False)


+-------------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
|filtered                                                                                               |label|prediction|probability                             |
+-------------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
|[, drain, weak, snap]                                                                                  |0    |0.0       |[0.9597066096293758,0.04029339037062418]|
|[, thumbs, seller]                                                                                     |1    |1.0       |[0.4013695429340416,0.5986304570659584] |
|[good, quality, bargain, bought, bought, cheapy, big, lots, sounded, awful, people, end, couldnt, hear]|1    |0.0       |[0.7179749204673925,0.28202507953260747]|
|[usable, keyboa

# The assignment should also include model evaluation including accuracy, precision, recall, and F1 score. 

In [8]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Initialize evaluators
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

# Compute metrics
accuracy = evaluator_accuracy.evaluate(predictions)
precision = evaluator_precision.evaluate(predictions)
recall = evaluator_recall.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

# Display results
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


Accuracy:  0.7099
Precision: 0.7102
Recall:    0.7099
F1 Score:  0.7087


In [9]:
from pyspark.sql.functions import col

# Group by actual and predicted label
confusion_df = predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
confusion_df.show()


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|   65|
|    0|       1.0|   20|
|    1|       0.0|   27|
|    1|       1.0|   50|
+-----+----------+-----+

